# [실습6] CLIP 멀티모달 패션 스타일 검색 시스템

AI 스타일리스트: 텍스트로 옷 찾기 🛍️

## 📚 실습 목표

이 실습에서는 **자연어로 옷을 검색**하는 실용적인 AI 시스템을 구축합니다:

1. **자연어 패션 검색** — "우아한 검은색 드레스", "편한 일상복" 같은 설명으로 옷 찾기
2. **CLIP 멀티모달 모델** — 이미지와 텍스트를 같은 벡터 공간에 표현
3. **스타일 추천 시스템** — 사용자의 스타일 설명에 맞는 옷 제안
4. **RAG 기초** — 검색 결과를 이용한 정보 제공 시스템으로의 확장

## 🔄 처리 파이프라인

```
[사용자 스타일 설명]
(자연어)
    ↓
[CLIP 텍스트 인코더]
    ↓
[512차원 텍스트 벡터]
    ↓
[의류 이미지 벡터들과 비교]
    ↓
[코사인 유사도 계산]
    ↓
[가장 어울리는 옷 TOP 5 추천]
    ↓
[추천 이유 + 스타일링 팁]
```

## 📖 핵심 개념

- **멀티모달 AI**: 텍스트와 이미지를 동시에 이해하는 AI
- **CLIP 모델**: "검은색 드레스" 텍스트와 실제 드레스 이미지를 같은 의미로 인식
- **벡터 공간의 의미**: 유사한 스타일은 벡터 공간에서도 가깝게 배치됨
- **패션 추천 시스템**: 온라인 쇼핑과 스타일링 앱의 핵심 기술

## 📝 과제 안내

이 노트북은 **실습 과제용**입니다. 아래 4곳에 `# TODO` 표시와 단계별 힌트만 남기고 구현을 비워두었습니다.

1. **텍스트 벡터화** (Step 2, 패션 스타일 프로필) — `fashion_profiles`를 CLIP 텍스트 인코더로 벡터화하는 반복문
2. **이미지 벡터화** (Step 2.6) — `fashion_images`를 CLIP 이미지 인코더로 벡터화하는 반복문
3. `search_by_image()` (Step 2.7) — 이미지로 유사한 패션 아이템 검색
4. `search_fashion_style()` (Step 2.7) — 텍스트 설명으로 유사한 패션 아이템 검색

각 TODO 주석의 순서대로 구현하면 이후 셀(Step 2.8 ~ Step 3)의 검색·시각화가 정상 동작합니다. 모델 로드, 데이터 다운로드, 시각화 코드는 이미 완성되어 있으니 그대로 사용하세요.

In [ ]:
%pip install -q torch torchvision
%pip install -q git+https://github.com/openai/CLIP.git
%pip install -q pillow requests matplotlib pandas numpy

%pip install -q torch torchvision
%pip install -q git+https://github.com/openai/CLIP.git
%pip install -q pillow requests matplotlib pandas numpy

# 한글 폰트 설정
import subprocess
import os

try:
    # 한글 폰트 파일 다운로드
    font_path = os.path.expanduser('~/.local/share/fonts/NotoSansCJK-Regular.ttc')
    if not os.path.exists(os.path.dirname(font_path)):
        os.makedirs(os.path.dirname(font_path), exist_ok=True)
    
    # 폰트 캐시 재구성
    import matplotlib.font_manager as fm
    fm.fontManager.addfont('/System/Library/Fonts/AppleGothic.ttf')
except:
    pass

# matplotlib 한글 설정
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

print('✅ 한글 폰트 설정 완료!')


## 🎯 CLIP으로 만드는 AI 스타일리스트

### 기존 검색의 문제점

```
❌ 기존 쇼핑 앱
입력: "드레스" 
결과: 모든 드레스 (10,000개) ← 너무 많음!

✅ CLIP 기반 검색
입력: "우아하고 세련된 검은 드레스"
결과: 딱 내 스타일의 드레스 (top 5) ← 정확함!
```

### CLIP이 하는 일

```
"검은색이고 
우아한 분위기의
긴 드레스"
        ↓
    [텍스트 이해]
        ↓
   실제 드레스 이미지와 매칭!
        ↓
   "어? 이 드레스 딱 내가
    원하는 스타일이네?"
```

### 실제 활용 예시

```
사용자: "직장에서 입을 수 있는 편안한 원피스"
  ↓
CLIP: 이 설명과 유사한 패션 이미지 검색
  ↓
추천: 심플한 검은색 셔츠 원피스 5개
  + 각 제품의 가격, 재질, 브랜드
  + 스타일링 팁 ("하얀색 블라우스와 잘 어울려요")
```

### CLIP과 패션의 완벽한 궁합

| 기술 | 패션에서의 활용 |
|------|-----------------|
| **텍스트 이해** | "시원한", "우아한", "캐주얼한" 같은 감정 표현 |
| **이미지 인식** | 의류의 색상, 패턴, 실루엣 정확히 파악 |
| **크로스모달** | "설명"과 "사진"이 서로 다른 형태지만 같은 의미 |
| **의미 검색** | 정확한 단어가 아닌 **스타일의 느낌**으로 검색 |

## 👔 Dataset: 패션 스타일 카테고리

오늘의 실습에서는 **8가지 패션 카테고리**를 다룹니다:

| 카테고리 | 설명 | 예시 스타일 |
|---------|------|----------|
| **상의 (Top)** | 셔츠, 티셔츠, 블라우스 | 클래식, 캐주얼, 오버사이즈 |
| **하의 (Bottom)** | 바지, 치마, 레깅스 | 슬림 핏, 와이드, 미니스커트 |
| **드레스 (Dress)** | 원피스, 미니드레스, 롱드레스 | 우아한, 보헤미안, 섹시 |
| **아우터 (Outer)** | 재킷, 코트, 후드집업 | 포멀, 캐주얼, 스포티 |
| **신발 (Shoes)** | 구두, 운동화, 부츠 | 클래식, 트렌디, 컴포트 |
| **액세서리 (Accessory)** | 가방, 목걸이, 모자 | 미니멀, 럭셔리, 비하인드 |
| **스타일링 세트 (Styling)** | 완성된 코디 | 데이트룩, 직장룩, 캐주얼룩 |
| **계절 룩 (Season)** | 계절별 컬렉션 | 여름, 겨울, 봄/가을 |

**실습의 특징**:
- 실제 온라인 쇼핑에 바로 적용 가능
- 개인 스타일 큐레이션의 원리 학습
- 패션 AI의 핵심 기술 경험

## 🧠 Step 1: CLIP 모델 로드 (AI 스타일리스트)

**OpenAI CLIP**: 400만 개의 이미지-텍스트 쌍으로 학습된 모델

학습 데이터에 포함된 것:
- ✅ 의류 사진 (셔츠, 바지, 드레스 등)
- ✅ 스타일 설명 ("우아한", "캐주얼한", "트렌디한")
- ✅ 색상, 패턴, 실루엣 정보
- ✅ 패션 관련 자연어 표현

따라서 패션 검색에 완벽하게 적용 가능합니다!

In [ ]:
# matplotlib 한글 폰트 설정
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'AppleGothic'  # macOS
plt.rcParams['axes.unicode_minus'] = False

import torch
import clip
from PIL import Image
import matplotlib.pyplot as plt
import requests
from tqdm import tqdm
import numpy as np

# CLIP 모델 로드
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/16", device=device)

print("=" * 50)
print("🛍️  AI 패션 스타일 추천 시스템 시작!")
print("=" * 50)
print(f"✅ CLIP model loaded!")
print(f"Device: {device}")
print(f"Model: ViT-B/16 (OpenAI)")
print(f"임베딩 차원: 512 (이미지 + 텍스트)")
print("=" * 50)

## 👗 Step 2: 패션 스타일 프로필 작성

각 패션 카테고리의 특징을 **자연어로 설명**합니다.

실제로는:
- 온라인 쇼핑몰의 상품 설명 사용 가능
- 스타일 블로거의 코디 설명 활용 가능
- 패션 매거진의 리뷰 텍스트 사용 가능

💡 **TIP**: 텍스트가 구체적일수록 더 정확한 매칭!

In [ ]:
# ===== 패션 스타일 프로필 =====

fashion_profiles = {
    "클래식 셔츠": "깔끔한 흰색이나 검은색 셔츠, 어떤 상황에도 잘 어울리는 기본 아이템",

    "캐주얼 티셔츠": "편한 코튼 티셔츠, 루즈한 핏, 캐주얼한 일상복",

    "우아한 드레스": "세련되고 검은색인 드레스, 미니멀한 디자인, 저녁 약속에 어울림",

    "오버사이즈 아우터": "넉넉한 핏의 현대적인 재킷이나 코트, 캐주얼한 분위기",

    "슬림 핏 바지": "몸에 딱 붙는 심플한 바지, 포멀한 상황에 적합",

    "와이드 팬츠": "편안한 넓은 바지, 트렌디하면서도 세련된 실루엣",

    "미니 스커트": "짧은 길이의 스커트, 젊고 활기찬 분위기, 여름 스타일",

    "롱 드레스": "발목까지 내려오는 긴 드레스, 우아하고 특별한 자리에 적합",

    "화이트 스니커즈": "깨끗한 흰색 운동화, 캐주얼하고 어떤 옷과도 잘 어울림",

    "부츠": "무릎까지 올라오는 부츠, 겨울 스타일, 세련된 분위기",

    "미니 크로스백": "작고 단정한 가방, 미니멀한 럭셔리 스타일",

    "빈티지 선글라스": "복고풍의 트렌디한 선글라스, 여름 필수 액세서리"
}

print("📦 12가지 패션 아이템을 로드했습니다!\n")
for item, description in list(fashion_profiles.items())[:3]:
    print(f"✓ {item}")
print(f"... 그외 {len(fashion_profiles) - 3}개\n")

# ===== 텍스트 벡터화 =====
fashion_vectors = {}
print("🔄 패션 아이템을 벡터화 중...")

with torch.no_grad():
    for fashion_item, description in fashion_profiles.items():
        # TODO: 아래 순서대로 구현하세요
        # 1. clip.tokenize(description)으로 토큰화하고 device로 옮기세요 (.to(device))
        # 2. model.encode_text()로 텍스트 벡터를 얻으세요
        # 3. 벡터를 L2 정규화하세요 (norm(dim=-1, keepdim=True)로 나누기)
        # 4. numpy 배열로 변환해 fashion_vectors[fashion_item]에 저장하세요 (.cpu().numpy().squeeze())
        pass

print("✅ 벡터화 완료!")
print(f"각 아이템: 512차원 벡터")

## 🌐 Step 2.5: Unsplash API로 실제 이미지 다운로드

**멀티모달 검색을 위해 각 패션 아이템의 실제 이미지가 필요합니다**

이미지 출처: Unsplash (무료 고품질 사진)
- API를 통해 각 패션 카테고리의 대표 사진 자동 다운로드
- 첫 실행만 시간이 걸림 (이후 로컬 캐시 사용)

In [ ]:
import requests
from pathlib import Path

UNSPLASH_API_KEY = "oqApddv5HPjxQrXCdrdt3QaDtPjITrZclAiuYEVPIE8"
UNSPLASH_API_URL = "https://api.unsplash.com"

CACHE_DIR = Path("./fashion_images")
CACHE_DIR.mkdir(exist_ok=True)

print("🌐 Unsplash API 설정 완료!")
print(f"캐시 디렉토리: {CACHE_DIR.absolute()}")

In [ ]:
def download_fashion_image(query, cache_name):
    """Unsplash에서 패션 이미지를 다운로드합니다."""
    cache_path = CACHE_DIR / f"{cache_name}.jpg"
    
    if cache_path.exists():
        return str(cache_path)
    
    params = {
        "query": query,
        "per_page": 1,
        "orientation": "portrait",
        "client_id": UNSPLASH_API_KEY
    }
    
    try:
        response = requests.get(f"{UNSPLASH_API_URL}/search/photos", params=params, timeout=10)
        response.raise_for_status()
        results = response.json()["results"]
        
        if results:
            image_url = results[0]["urls"]["regular"]
            img_response = requests.get(image_url, timeout=10)
            with open(cache_path, 'wb') as f:
                f.write(img_response.content)
            return str(cache_path)
    except Exception as e:
        print(f"❌ {cache_name}: {str(e)[:40]}")
    
    return None

print("✅ 이미지 다운로드 함수 준비 완료")

In [ ]:
image_queries = {
    "클래식 셔츠": "classic white shirt fashion woman",
    "캐주얼 티셔츠": "casual t-shirt woman style",
    "우아한 드레스": "elegant black dress woman",
    "오버사이즈 아우터": "oversized jacket coat fashion",
    "슬림 핏 바지": "slim fit pants fashion woman",
    "와이드 팬츠": "wide pants fashion woman",
    "미니 스커트": "mini skirt fashion style",
    "롱 드레스": "long dress elegant woman",
    "화이트 스니커즈": "white sneakers fashion",
    "부츠": "boots fashion woman",
    "미니 크로스백": "mini crossbody bag fashion",
    "빈티지 선글라스": "vintage sunglasses fashion"
}

print("📸 12가지 패션 아이템의 이미지를 다운로드 중...\\n")
fashion_images = {}

for item_name, query in image_queries.items():
    cache_name = item_name.replace(" ", "_")
    image_path = download_fashion_image(query, cache_name)
    fashion_images[item_name] = image_path
    
    if image_path:
        print(f"✅ {item_name:15s} → 다운로드 완료")
    else:
        print(f"❌ {item_name:15s} → 실패")

print(f"\\n🎉 총 {len([v for v in fashion_images.values() if v])}개 이미지 준비 완료!")

## 🖼️ Step 2.6: 이미지를 벡터로 변환

**CLIP의 이미지 인코더 사용**

멀티모달의 핵심:
- 텍스트 벡터와 이미지 벡터가 같은 512차원 공간에 있음
- "검은색 드레스" 텍스트와 검은 드레스 이미지가 유사한 방향

In [ ]:
print("🔄 12가지 패션 아이템의 이미지를 벡터로 변환 중...\\n")

image_vectors = {}
valid_images = {}

with torch.no_grad():
    for item_name, image_path in fashion_images.items():
        if image_path is None:
            continue
        
        try:
            # TODO: 아래 순서대로 구현하세요
            # 1. PIL로 이미지를 열고 RGB로 변환하세요 (Image.open, .convert('RGB'))
            # 2. preprocess()로 전처리 후 배치 차원을 추가하고 device로 옮기세요 (.unsqueeze(0), .to(device))
            # 3. model.encode_image()로 이미지 벡터를 얻으세요
            # 4. 벡터를 L2 정규화하세요 (norm(dim=-1, keepdim=True)로 나누기)
            # 5. numpy 배열로 변환해 image_vectors[item_name]에 저장하세요 (.cpu().numpy().squeeze())
            # 6. valid_images[item_name]에 image_path를 저장하고, 완료 메시지를 출력하세요
            #    (print(f"✅ {item_name:15s} → 벡터화 완료"))
            pass
        except Exception as e:
            print(f"❌ {item_name:15s} → {str(e)[:40]}")

print(f"\\n🎉 총 {len(image_vectors)}개 이미지 벡터화 완료!")

## 🔍 Step 2.7: 멀티모달 검색 함수

텍스트 검색 + 이미지 기반 검색을 모두 지원합니다.

In [ ]:
def search_by_image(image_path, image_vectors, valid_images, n_results=5):
    """이미지로 유사한 패션 아이템을 검색합니다."""
    # TODO: 아래 순서대로 구현하세요
    # 1. try 블록 안에서 Image.open(image_path).convert('RGB')로 쿼리 이미지를 열고
    #    preprocess()로 전처리 후 배치 차원을 추가하고 device로 옮기세요 (.unsqueeze(0), .to(device))
    # 2. torch.no_grad() 안에서 model.encode_image()로 벡터를 얻고 L2 정규화하세요
    #    (norm(dim=-1, keepdim=True)로 나누기), numpy로 변환하세요 (.cpu().numpy().squeeze())
    #    실패 시 except Exception as e: 에러 메시지 출력 후 [] 를 반환하세요
    # 3. image_vectors에 저장된 모든 아이템과 코사인 유사도(@ 연산자, 내적)를 계산해
    #    (item_name, similarity, img_path) 튜플 리스트를 만드세요 (img_path는 valid_images[item_name])
    # 4. 유사도 내림차순으로 정렬 후 상위 n_results개를 반환하세요
    pass

print("✅ 멀티모달 검색 함수 준비 완료!")

In [ ]:
def search_fashion_style(style_description, fashion_vectors, n_results=5):
    """
    사용자의 스타일 설명으로 옷을 검색합니다.
    
    Args:
        style_description: 원하는 스타일 설명 (예: "우아한 검은 드레스")
        fashion_vectors: 패션 아이템 벡터들
        n_results: 추천할 아이템 개수
    
    Returns:
        [(아이템명, 유사도 점수), ...] 정렬된 리스트
    """
    # TODO: 아래 순서대로 구현하세요
    # 1. torch.no_grad() 안에서 clip.tokenize(style_description)으로 토큰화하고 device로 옮기세요 (.to(device))
    # 2. model.encode_text()로 벡터를 얻고 L2 정규화하세요 (norm(dim=-1, keepdim=True)로 나누기)
    #    numpy 배열로 변환하세요 (.cpu().numpy().squeeze())
    # 3. fashion_vectors에 저장된 모든 아이템과 코사인 유사도(@ 연산자, 내적)를 계산해
    #    (fashion_item, similarity) 튜플 리스트를 만드세요
    # 4. 유사도 내림차순으로 정렬 후 상위 n_results개를 반환하세요
    pass

# ===== 테스트 =====
style_query = "우아하고 세련된 검은색 옷이 필요해"
results = search_fashion_style(style_query, fashion_vectors, n_results=5)

print(f"\n🛍️  당신의 스타일: '{style_query}'\n")
print("💁 추천 스타일링 (유사도 순):")
print("-" * 50)
for i, (item, similarity) in enumerate(results, 1):
    score = "⭐" * int(similarity * 5)
    print(f"{i}. {item:20s} | {similarity:.2%} | {score}")
print("-" * 50)

## ✨ Step 2.8: 멀티모달 검색 테스트

텍스트 검색과 이미지 검색을 비교해봅시다!

In [ ]:
# ===== 텍스트 검색 vs 이미지 검색 비교 =====

print("\\n" + "="*70)
print("🎯 텍스트 검색 vs 이미지 기반 멀티모달 검색")
print("="*70)

# 🔤 텍스트 검색
query_text = "우아하고 세련된 검은색 옷"
results_text = search_fashion_style(query_text, fashion_vectors, n_results=12)

print(f"\\n🔤 텍스트 검색: '{query_text}'")
print("-" * 70)
for i, (item, sim) in enumerate(results_text, 1):
    star = "⭐" * int(sim * 5)
    print(f"{i:2d}. {item:20s} | {sim:.1%} | {star}")


# 🖼️ 이미지 검색
if valid_images.get("우아한 드레스"):
    results_image = search_by_image(valid_images["우아한 드레스"], image_vectors, valid_images, n_results=12)
    
    print(f"\\n🖼️  이미지 기반 멀티모달 검색: 우아한 드레스 사진으로 검색")
    print(f"    (유사도 높은 순서대로 배치됨)")
    print("-" * 70)
    for i, (item, sim, _) in enumerate(results_image, 1):
        star = "⭐" * int(sim * 5)
        print(f"{i:2d}. {item:20s} | {sim:.1%} | {star}")
    
    # 🎨 시각화: 6행 2열 (더 큰 이미지)
    fig, axes = plt.subplots(6, 2, figsize=(12, 20))
    axes = axes.flatten()
    
    # 12개 결과 이미지 (유사도 높은 순서대로)
    for idx, (item, sim, img_path) in enumerate(results_image):
        result_img = Image.open(img_path).convert('RGB')
        axes[idx].imshow(result_img)
        
        # 제목: 순위 + 아이템명 + 유사도
        rank = idx + 1
        title_text = f'#{rank} {item}\\n{sim:.1%}'
        fontweight = 'bold' if sim > 0.95 else 'normal'
        color = 'darkred' if sim > 0.95 else 'black'
        axes[idx].set_title(title_text, fontsize=12, weight=fontweight, color=color)
        axes[idx].axis('off')
    
    plt.suptitle('🖼️ 멀티모달 검색: 우아한 드레스 이미지로 검색한 12개 패션 아이템\\n(유사도 높은 순서대로 배치)',
                 fontsize=14, weight='bold', y=0.995)
    plt.tight_layout()
    plt.show()
    
    print("\\n💡 멀티모달 검색의 원리:")
    print(f"  1️⃣ 쿼리 이미지 → CLIP 이미지 인코더 → 512차원 벡터")
    print(f"  2️⃣ 모든 패션 아이템 이미지 → 512차원 벡터")
    print(f"  3️⃣ 코사인 유사도로 가장 비슷한 순서대로 정렬")
    print(f"  4️⃣ 최고 매칭: {results_image[0][0]} ({results_image[0][1]:.1%})")
    print(f"  5️⃣ CLIP이 이미지의 '우아함', '색감', '실루엣' 등을 모두 이해함")

## 🎯 Step 2.9: 다양한 키워드로 멀티모달 검색

**패션 스타일을 다양한 방식으로 검색해봅시다!**

In [ ]:
# 다양한 시나리오별 멀티모달 검색

scenarios = [
    {
        'text': '우아하고 세련된 검은 옷',
        'image': '우아한 드레스',
        'emoji': '🎩',
        'description': '정장 같은 우아한 스타일'
    },
    {
        'text': '편하고 캐주얼한 일상복',
        'image': '캐주얼 티셔츠',
        'emoji': '👕',
        'description': '편안한 일상 스타일'
    },
    {
        'text': '트렌디하고 활기찬 여름 스타일',
        'image': '미니 스커트',
        'emoji': '☀️',
        'description': '시원한 여름 스타일'
    },
    {
        'text': '따뜻하고 세련된 겨울 분위기',
        'image': '부츠',
        'emoji': '❄️',
        'description': '겨울 감성 스타일'
    },
]

print('\\n' + '='*80)
print('🎨 다양한 상황별 멀티모달 검색 비교')
print('='*80)

for scenario_idx, scenario in enumerate(scenarios, 1):
    text_query = scenario['text']
    image_item = scenario['image']
    emoji = scenario['emoji']
    description = scenario['description']
    
    print(f'\\n{emoji} [{scenario_idx}] {description}')
    print('-' * 80)
    
    # 텍스트 검색
    text_results = search_fashion_style(text_query, fashion_vectors, n_results=5)
    print(f'\\n🔤 텍스트: "{text_query}"')
    for i, (item, sim) in enumerate(text_results, 1):
        print(f'   {i}. {item:20s} ({sim:.1%})')
    
    # 이미지 검색
    if valid_images.get(image_item):
        image_results = search_by_image(valid_images[image_item], image_vectors, valid_images, n_results=5)
        print(f'\\n🖼️  이미지: "{image_item}" 사진으로 검색')
        for i, (item, sim, _) in enumerate(image_results, 1):
            print(f'   {i}. {item:20s} ({sim:.1%})')
        
        # 시각화: 3행 2열 (쿼리 + 5개 결과)
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        # 쿼리 이미지
        query_img = Image.open(valid_images[image_item]).convert('RGB')
        axes[0].imshow(query_img)
        axes[0].set_title(f'🔍 쿼리\\n{image_item}', fontsize=11, weight='bold', color='red')
        axes[0].axis('off')
        
        # 검색 결과 5개
        for idx, (item, sim, img_path) in enumerate(image_results):
            ax_idx = idx + 1
            result_img = Image.open(img_path).convert('RGB')
            axes[ax_idx].imshow(result_img)
            axes[ax_idx].set_title(f'{item}\\n{sim:.1%}', fontsize=10, weight='bold')
            axes[ax_idx].axis('off')
        
        title = f'{emoji} {description} - 멀티모달 검색 결과'
        plt.suptitle(title, fontsize=13, weight='bold')
        plt.tight_layout()
        plt.show()

print('\\n' + '='*80)
print('💡 멀티모달 검색의 특징:')
print('='*80)
print('  • 텍스트와 이미지 모두 같은 벡터 공간에 존재')
print('  • "우아함", "캐주얼함" 같은 추상적 개념을 이해')
print('  • 사진만으로도 비슷한 스타일의 옷을 찾을 수 있음')
print('  • 온라인 쇼핑에 바로 적용 가능한 기술')
print('  • RAG(Retrieval-Augmented Generation)의 기초')


## 🔍 Step 3: 사용자 스타일 설명으로 검색

**AI 스타일리스트의 작동 원리**:

```
👤 사용자: "오늘은 직장 가는데 깔끔하고 세련된 룩이 필요해"
    ↓
🤖 CLIP: 이 설명을 벡터로 변환
    ↓
📊 유사도 계산: 
   - 우아한 드레스: 0.87 ⭐⭐⭐
   - 클래식 셔츠: 0.82 ⭐⭐
   - 슬림 핏 바지: 0.79 ⭐⭐
   - 캐주얼 티셔츠: 0.45 (안 어울림)
   ↓
💁 추천: "우아한 드레스 + 클래식 셔츠 + 화이트 스니커즈" 코디 제안
```

**검색이 정확한 이유**:
- CLIP은 "깔끔하다", "세련되다" 같은 추상적인 표현 이해
- "직장", "업무 환경" 같은 맥락 파악
- 텍스트의 감정과 이미지의 분위기 매칭

In [ ]:
# ===== 다양한 상황별 스타일 검색 =====

style_scenarios = [
    ("오늘은 편하고 편안한 일상복이 필요해", "🏠 집에서 편하게 입을 룩"),
    ("직장 면접이 있어서 깔끔하고 신뢰감 있는 이미지 필요", "💼 면접 룩"),
    ("친구들과 카페 가는데 트렌디하고 캐주얼한 스타일로", "☕ 데이트/카페 룩"),
    ("여름 날씨에 시원하고 밝은 분위기의 옷", "☀️ 여름 룩"),
    ("밤새 편하고 부드러운 느낌의 옷", "🌙 편한 룩"),
]

print("\n📋 다양한 상황별 스타일 추천\n")
print("=" * 70)

for scenario, emoji in style_scenarios:
    results = search_fashion_style(scenario, fashion_vectors, n_results=3)
    
    print(f"\n{emoji}")
    print(f"상황: {scenario}")
    print("-" * 70)
    
    for i, (item, similarity) in enumerate(results, 1):
        print(f"  {i}순위: {item:20s} (유사도: {similarity:.1%})")

## 📝 실습 정리

### 🎯 배운 내용

1. **멀티모달 AI의 실제 활용**: 패션 추천의 핵심 기술
2. **CLIP 모델의 능력**: 텍스트로 이미지 검색
3. **벡터 공간의 의미**: 유사한 스타일이 벡터 공간에서도 가까움
4. **자연어 이해**: "우아한", "캐주얼한" 같은 추상적 표현 처리

### 💡 실제 적용 사례

이 기술은 이미 다음에서 사용 중:
- 🛍️ **온라인 쇼핑몰**: "이 옷 같은 스타일" 검색
- 👗 **패션 앱**: 스타일 큐레이션 및 추천
- 💄 **뷰티 플랫폼**: 메이크업 및 패션 매칭
- 📸 **SNS**: 인플루언서의 스타일과 유사한 상품 찾기

### 🚀 확장 아이디어

```python
# 1. 실제 온라인 쇼핑몰 API와 연결
# - Naver Shopping, Coupang 상품과 매칭
# - 실시간 가격 정보 추가

# 2. 개인화된 추천
# - 사용자의 구매 이력 학습
# - 선호 색상, 사이즈 자동 반영

# 3. 이미지 기반 검색과 통합
# - SNS에서 찍은 옷 사진
# - "이 옷 같은 스타일 찾아줘"

# 4. 가상 피팅 (AR/VR)
# - 실제 옷을 입어보지 않고 미리 보기
```

### 🎓 다음 강의로의 연결

이 실습6 → **실습7(RAG)**로 확장:

```
사용자 입력: "검은색이고 우아한 드레스가 필요한데, 
             50만원 이하이고 면 재질로 찾아줄래?"
    ↓
[CLIP 검색] (이미 배움!)
    ↓
[검색 결과] + [상품 정보 DB]
    ↓
[LLM 생성] ← 다음 강의!
    ↓
답변: "이 드레스를 추천합니다!
      - 가격: 45만원
      - 재질: 100% 순면
      - 스타일링 팁: 화이트 블라우스와 함께 입으면..."
```

## ❓ 자주 묻는 질문

### Q1: 패션 말고 다른 도메인도 가능한가?
**A**: 완전히 가능합니다!
- 가구/인테리어: "미니멀한 북유럽 스타일"
- 자동차: "세련된 스포츠카"
- 뷰티: "자연스러운 페이스 메이크업"
- 음악: "신나는 포크음악" (커버 아트 기반)

### Q2: 실제 쇼핑몰과 연결하려면?
**A**: 3단계:
```python
# 1. 쇼핑몰 API에서 상품 이미지 가져오기
products = naver_shopping_api.search("원피스")

# 2. CLIP으로 상품 벡터화
product_vectors = clip.encode_image(products)

# 3. 사용자 검색과 매칭
results = search_products(user_query, product_vectors)
```

### Q3: 정확도를 높이려면?
**A**: 
- 더 상세한 설명 사용 ("깔끔한" → "깔끔하고 세련된 무채색")
- 더 큰 CLIP 모델 (ViT-L/14)
- 패션 특화 파인튜닝

### Q4: 성능은?
**A**: 충분히 빠릅니다!
- 12개 아이템 검색: < 10ms
- 1000개 아이템 검색: < 50ms
- 온라인 쇼핑에 즉시 적용 가능

---

# 🎓 학습 요약

| 단계 | 학습 내용 | 핵심 기술 |
|------|-----------|-----------|
| 1 | CLIP 개념 | 이미지-텍스트 공유 임베딩 공간 |
| 2 | 모델 로드 | HuggingFace transformers, CLIPModel |
| 3 | 텍스트 검색 | 텍스트 쿼리 → 이미지 유사도 검색 |
| 4 | 이미지 검색 | 이미지 → 유사 이미지 검색 |
| 5 | 패션 응용 | 스타일 추천, 크로스모달 검색 |

## 핵심 개념 정리

- **CLIP**: OpenAI가 개발한 멀티모달 모델. 이미지와 텍스트를 **같은 벡터 공간**에 매핑하여 교차 검색 가능
- **공유 임베딩 공간**: "빨간 드레스" 텍스트 벡터와 빨간 드레스 이미지 벡터가 가까이 위치
- **코사인 유사도**: 벡터 간 방향 유사성으로 이미지-텍스트 관련도 측정
- **Zero-shot 검색**: 학습하지 않은 새로운 카테고리도 텍스트 설명만으로 검색 가능

## ✅ 실습 체크리스트

- [ ] CLIP이 이미지와 텍스트를 같은 공간에 매핑하는 원리를 이해한다
- [ ] 텍스트 쿼리로 이미지를 검색할 수 있다
- [ ] 이미지 쿼리로 유사 이미지를 검색할 수 있다
- [ ] 코사인 유사도의 의미를 설명할 수 있다
- [ ] 멀티모달 검색의 실무 활용 사례를 알고 있다

## 🚀 다음 단계

- **실습 7**: CLIP 검색 기능을 FastAPI로 배포하여 실제 서비스를 구축합니다
- **심화**: CLIP Fine-tuning(도메인 특화), FAISS와 연동하여 대규모 인덱싱, 실시간 추천 시스템
- **응용**: 의료 영상 검색, 제품 카탈로그 검색, 소셜 미디어 콘텐츠 분류